# Week 5: National Gap Survey

**NWR Coverage Gap Research — Summer 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/W2NJL/nwr-gap-research/blob/main/week5_national_survey.ipynb)

---

Week 4 gave you a complete picture of one state. Week 5 scales to the whole country.

Querying every US city would take days of API calls, so you'll first design a **sampling strategy** that gives national coverage while remaining runnable in a single session. You'll then use those results to:
- Build a national coverage map
- Rank states by gap severity
- Compute a **gap priority score** that weights both population and signal depth
- Identify the top 50 most at-risk locations that will anchor the formal report in Weeks 6--7

---
## Part 1: Setup

In [ ]:
!pip install folium openpyxl --quiet

import requests, json, io, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from IPython.display import display

WX_URL     = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/wx_stations.csv"
CITIES_URL = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/us_cities.csv"
RUCC_URL   = "https://www.ers.usda.gov/webdocs/DataFiles/53251/ruralurbancontinuumcodes2023.xlsx"

wx     = pd.read_csv(WX_URL)
wx     = wx[wx['country'] == 'USA'].dropna(subset=['latitude', 'longitude']).copy()
wx['longitude'] = wx['longitude'] * -1
cities = pd.read_csv(CITIES_URL)

COVERAGE_THRESHOLD = 50.0
RADIOLAND_BASE     = "http://52.151.197.43/search_stream"

print(f"wx_stations: {len(wx)} US transmitters")
print(f"cities:      {len(cities)} US cities")

In [ ]:
def query_nwr_coverage(lat, lon, rx_height=10, min_sig_strength=3, verbose=False):
    """
    Query the RadioLand API for NWR stations receivable at (lat, lon).
    Returns a DataFrame sorted by field_strength descending, or None on failure.
    """
    params = {
        "lat": lat,
        "lon": lon,
        "search_freq": "none",
        "callsign": "none",
        "request_type": 1,
        "pi_code": "none",
        "sig_strength": min_sig_strength,
        "am_sig_strength": 2,
        "startMiles": "none",
        "miles": "null",
        "slogan": "none",
        "owner": "none",
        "format": "none",
        "wfo": "none",
        "rxHeight": rx_height,
        "mlbTeam": "none",
        "market": "none",
        "country": "none",
        "sp": "none",
        "measurementUnit": "metric",
        "locationName": "",
        "broadcastBand": "WX",
        "timeOfDay": "day",
        "model": "ml",
    }
    try:
        resp = requests.get(RADIOLAND_BASE, params=params, stream=True, timeout=90)
        resp.raise_for_status()
    except requests.RequestException as e:
        if verbose:
            print(f"Request failed: {e}")
        return None

    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode('utf-8')
        if not line.startswith('data: '):
            continue
        event = json.loads(line[6:])
        if event.get('type') == 'progress' and verbose:
            print(f"  [{event['percentage']:>3}%] {event['message']}", end='\r')
        elif event.get('type') == 'complete':
            if verbose:
                print("  [100%] Done.                                        ")
            result = json.loads(event['data'])
            df = pd.DataFrame(result['data'])
            if df.empty:
                return df
            for col in ['lon', 'transmitter_lon']:
                if col in df.columns:
                    df[col] = -df[col].abs()
            return df.sort_values('field_strength', ascending=False).reset_index(drop=True)
    return None


def get_best_signal(lat, lon, threshold=COVERAGE_THRESHOLD):
    df = query_nwr_coverage(lat, lon)
    if df is None or df.empty:
        return {'best_callsign': None, 'best_field_strength': 0.0,
                'best_distance_km': None, 'stations_above_threshold': 0, 'covered': False}
    best = df.iloc[0]
    return {
        'best_callsign':            best['callsign'],
        'best_field_strength':      float(best['field_strength']),
        'best_distance_km':         float(best.get('distance', float('nan'))),
        'stations_above_threshold': int((df['field_strength'] >= threshold).sum()),
        'covered':                  float(best['field_strength']) >= threshold,
    }

---
## Part 2: Sampling Strategy

Querying all cities in our dataset at 20 seconds per call would take **90+ hours**. We need a smarter approach.

Two common strategies:

**Strategy A -- Population cutoff:** Query every city above some population threshold (e.g., pop >= 5,000). This prioritizes where people live but may miss sparsely populated rural states entirely.

**Strategy B -- Stratified state sample:** Take the top N cities by population *per state*, ensuring every state gets represented even if it has few large cities.

There is no single correct answer -- it depends on what you're optimizing for. This is a real research design decision.

In [ ]:
print("Population distribution:")
print(cities['population'].describe(percentiles=[.25, .5, .75, .9, .95, .99]))
print()

print("Cities above common thresholds:")
for cutoff in [1000, 2500, 5000, 10000, 25000]:
    n = (cities['population'] >= cutoff).sum()
    print(f"  pop >= {cutoff:>6,}: {n:>5,} cities  (~{n*20/3600:.1f} hrs at 20s/query)")

### E1 -- Design Your Sampling Strategy

Choose a strategy (or a hybrid) and implement it below to create a DataFrame called `sample_cities`. Aim for **300--600 cities** -- enough for national patterns, manageable to run in a few hours.

Requirements:
- Must include cities from all 48 contiguous states (plus DC if your dataset includes it)
- Must include at least a few smaller or rural cities per state, not only large metros
- Should total 300--600 rows

In the markdown cell below, document your approach and justify the tradeoffs.

**Your strategy:**

*(Describe your sampling approach here. Example: "I took the 8 largest cities per state plus the 2 smallest cities per state, to capture both urban coverage and rural gaps in each state. This gives ~500 cities and ensures every state is represented.")*

In [ ]:
# YOUR CODE HERE -- build sample_cities
# Must have columns: city, state_id, lat, lng, population, county_name
#
# Example approaches:
#
# Top N + bottom N per state:
#   tops = cities.groupby('state_id', group_keys=False).apply(lambda g: g.nlargest(8, 'population'))
#   bots = cities.groupby('state_id', group_keys=False).apply(lambda g: g.nsmallest(2, 'population'))
#   sample_cities = pd.concat([tops, bots]).drop_duplicates(subset=['city', 'state_id'])
#
# Population cutoff:
#   sample_cities = cities[cities['population'] >= 10000].copy()
#
# YOUR CODE:


print(f"Sample size: {len(sample_cities)} cities across {sample_cities['state_id'].nunique()} states")
sample_cities[['city', 'state_id', 'population']].sample(min(10, len(sample_cities)))

---
## Part 3: The National Sweep

This will take several hours to run. A percentage counter helps you track progress. Save to CSV immediately when done -- if the session disconnects, you can reload and skip already-queried cities.

In [ ]:
results = []
total   = len(sample_cities)

for i, row in sample_cities.reset_index(drop=True).iterrows():
    label = f"{row['city']}, {row['state_id']}"
    pct   = (i + 1) / total * 100
    print(f"[{i+1}/{total} | {pct:.1f}%] {label}...", end=' ', flush=True)

    summary = get_best_signal(row['lat'], row['lng'])
    summary.update({
        'city':        row['city'],
        'state_id':    row['state_id'],
        'lat':         row['lat'],
        'lng':         row['lng'],
        'population':  row['population'],
        'county_name': row.get('county_name', ''),
    })
    results.append(summary)

    sig    = summary['best_field_strength']
    status = 'COVERED' if summary['covered'] else 'GAP'
    print(f"{status} ({sig:.1f} dB)")
    time.sleep(2)

national_df = pd.DataFrame(results)
national_df.to_csv("national_coverage.csv", index=False)
print(f"\nDone. Saved {len(national_df)} rows to national_coverage.csv")
print(f"Covered: {national_df['covered'].sum()} | Gaps: {(~national_df['covered']).sum()}")

In [ ]:
# Reload without re-running:
# national_df = pd.read_csv("national_coverage.csv")
# print(f"Loaded {len(national_df)} rows")

---
## Part 4: National Coverage Map

### E3 -- Build the National Map

Scale your Week 4 map to the full country. Adjustments for national scale:
- Center at ~39.5N, 98.4W; `zoom_start=4`
- Smaller markers -- at this scale, `radius=4` and `fill_opacity=0.5` keep it readable
- Tile: `CartoDB positron` (clean background, NWS-style)

In [ ]:
# YOUR CODE HERE

m = folium.Map(location=[39.5, -98.4], zoom_start=4, tiles='CartoDB positron')

for _, row in national_df.iterrows():
    color = 'green' if row['covered'] else 'red'
    popup = folium.Popup(
        f"<b>{row['city']}, {row['state_id']}</b><br>"
        f"Signal: {row['best_field_strength']:.1f} dB<br>"
        f"Station: {row['best_callsign']}<br>"
        f"Pop: {int(row['population']):,}<br>"
        f"{'COVERED' if row['covered'] else 'GAP'}",
        max_width=200
    )
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.5,
        popup=popup
    ).add_to(m)

display(m)

**Map Questions:**

1. Visually, which region of the US looks most underserved?
2. Are there geographic patterns you did not expect?
3. Does the density of red vs. green points match your intuition about where NOAA has invested in transmitter infrastructure?

**Your answers:**

1.
2.
3.

---
## Part 5: State-by-State Breakdown

### E4 -- Gap Rates by State

Which states have the highest coverage gap rates in your sample? Compute and visualize.

Keep in mind: these rates reflect *your sample*, not all cities in each state. Your sampling strategy affects these numbers.

In [ ]:
# YOUR CODE HERE
# 1. Group by state_id: city count, gap count, gap rate, avg field strength, total gap population
# 2. Sort by gap_rate descending
# 3. Plot a horizontal bar chart of the top 15 states by gap rate

state_summary = (national_df
    .groupby('state_id')
    .agg(
        cities_sampled = ('city', 'count'),
        gap_cities     = ('covered', lambda x: (~x).sum()),
        total_gap_pop  = ('population', lambda x: x[~national_df.loc[x.index, 'covered']].sum()),
        avg_signal     = ('best_field_strength', 'mean'),
    )
    .assign(gap_rate=lambda d: d['gap_cities'] / d['cities_sampled'])
    .sort_values('gap_rate', ascending=False)
)

print("Top 15 states by gap rate (sampled cities):")
print(state_summary.head(15).to_string())

fig, ax = plt.subplots(figsize=(10, 7))
top15 = state_summary.head(15)
ax.barh(top15.index[::-1], top15['gap_rate'][::-1] * 100, color='tomato')
ax.axvline(state_summary['gap_rate'].mean() * 100, color='black',
           linestyle='--', label=f"Sample avg: {state_summary['gap_rate'].mean():.1%}")
ax.set_xlabel('Gap Rate (%)')
ax.set_title('Top 15 States by NWR Coverage Gap Rate (Sampled Cities)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 6: Gap Priority Scoring

Identifying gaps is step one. Prioritizing *which gaps to address first* is step two -- and requires a metric that accounts for both how bad the coverage is and how many people are affected.

### The Gap Priority Score

```
gap_depth      = max(0,  COVERAGE_THRESHOLD - best_field_strength)
priority_score = population * gap_depth
```

- A city of 50,000 with 0 dB has higher priority than a city of 500 with 45 dB.
- `gap_depth` = 0 for covered cities, so they drop out automatically.

In Weeks 6--7 you'll refine this by also incorporating **hazard exposure** (Storm Events) and **demographic vulnerability** (RUCC / Census data).

In [ ]:
national_df['gap_depth']      = (COVERAGE_THRESHOLD - national_df['best_field_strength']).clip(lower=0)
national_df['priority_score'] = national_df['population'] * national_df['gap_depth']

# Zero out covered cities
national_df.loc[national_df['covered'], ['gap_depth', 'priority_score']] = 0

print("Priority score stats (gap cities only):")
print(national_df[national_df['priority_score'] > 0]['priority_score'].describe())

### E5 -- Visualize Priority Scores

Plot the distribution of priority scores for gap cities. Use a log scale -- these values span many orders of magnitude.

In [ ]:
# YOUR CODE HERE
# Histogram of priority_score for gap cities (priority_score > 0)
# Use ax.set_xscale('log')
# Add a vertical line at the median

gap_only = national_df[national_df['priority_score'] > 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(gap_only['priority_score'], bins=40, color='tomato', edgecolor='white')
ax.axvline(gap_only['priority_score'].median(), color='black', linestyle='--',
           label=f"Median: {gap_only['priority_score'].median():,.0f}")
ax.set_xscale('log')
ax.set_xlabel('Priority Score (population x gap depth) -- log scale')
ax.set_ylabel('Number of Gap Cities')
ax.set_title('Distribution of NWR Gap Priority Scores')
ax.legend()
plt.tight_layout()
plt.show()

**Questions:**

1. Is the distribution symmetric on the log scale, or skewed? What does that suggest about how concentrated the worst gaps are?
2. What is one weakness of the `population * gap_depth` formula? What does it fail to capture?
3. Propose one variable you could add to the score. Where would you get the data, and why would it improve the ranking?

**Your answers:**

1.
2.
3.

---
## Part 7: Top 50 Most At-Risk Locations

### E6 -- Rank and Display

Generate the top 50 gap cities by priority score. This list will anchor the formal gap report in Weeks 6--7.

In [ ]:
top50 = (national_df[national_df['priority_score'] > 0]
         .nlargest(50, 'priority_score')
         [['city', 'state_id', 'population', 'best_field_strength',
           'gap_depth', 'priority_score', 'best_callsign', 'county_name']]
         .reset_index(drop=True))

top50.index += 1   # 1-based ranking
print("Top 50 Most At-Risk NWR Gap Locations")
print(f"Score = population x (50 - best_field_strength)")
print()
print(top50.to_string())

In [ ]:
# Focused map: all sampled cities in gray, top 50 highlighted in orange

m2 = folium.Map(location=[39.5, -98.4], zoom_start=4, tiles='CartoDB positron')

for _, row in national_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=3, color='gray', fill=True, fill_opacity=0.25
    ).add_to(m2)

for rank, row in top50.iterrows():
    match = national_df[national_df['city'] == row['city']]
    if match.empty:
        continue
    lat = match.iloc[0]['lat']
    lng = match.iloc[0]['lng']
    popup = folium.Popup(
        f"<b>#{rank} {row['city']}, {row['state_id']}</b><br>"
        f"Pop: {int(row['population']):,}<br>"
        f"Signal: {row['best_field_strength']:.1f} dB (gap depth: {row['gap_depth']:.1f})<br>"
        f"Priority: {row['priority_score']:,.0f}",
        max_width=220
    )
    folium.CircleMarker(
        location=[lat, lng],
        radius=9, color='darkorange', fill=True, fill_opacity=0.85, popup=popup
    ).add_to(m2)

display(m2)

**Questions:**

1. Are the highest-priority gaps concentrated in the states that topped your state-by-state bar chart?
2. Are there any cities in the top 50 with relatively small populations but extreme gap depth? What does that tell you about transmitter coverage in those areas?
3. Imagine presenting your #1 city to a FEMA program manager. Write two sentences explaining why it tops the list and what it would take to fix it.

**Your answers:**

1.
2.
3.

---
## Week 5 Reflection

1. Your national survey is based on a sample. How confident are you that the top 50 list reflects reality? What would change if you doubled the sample size?
2. The priority formula is `population * gap_depth`. In Weeks 6--7 you'll layer on hazard exposure data. Which hazard type do you think will most dramatically reshuffle the rankings, and why?
3. You now have a full pipeline: station data (Week 1) > external datasets (Week 2) > API queries (Week 3) > state sweep (Week 4) > national survey (Week 5). What is the weakest link in that pipeline, and what would you do to strengthen it?
4. What is one question your data raises that you'd want answered before submitting the final report to FEMA?

**Your answers:**

1.
2.
3.
4.